# TP1 — ¿Qué es un mercado en este problema?

**Mentorería DiploDatos 2026** · Proyecto Final: Detección de Ofertas Hoteleras ID90Travel

**Fecha de entrega:** 31/07

---

## Objetivo

Antes de intentar detectar si un precio es una oferta, hay que resolver una pregunta más básica: **¿contra qué se compara ese precio?**

La respuesta no es obvia. No alcanza con decir "contra otros precios del mismo destino" — porque los precios en Las Vegas un viernes de diciembre no tienen nada que ver con los de un martes de febrero, ni los de un hotel económico con los de uno de lujo.

El TP1 es una **exploración abierta del dataset** para entender qué dimensiones producen grupos de observaciones donde los precios se comportan de forma homogénea — es decir, donde comparar tiene sentido.

## Preguntas de investigación

### 1. Definición de mercado: ¿contra qué se compara?

**El problema de la referencia**: para saber si un precio es una oferta necesitás un precio de referencia del mismo mercado. Un hotel en Las Vegas no compite con uno en Orlando — aunque ambos sean destinos turísticos norteamericanos. Pero, ¿Las Vegas y Henderson (ciudad a 30 km) son el mismo mercado? ¿Y Las Vegas y Miami?

**La pregunta de fondo es: ¿qué define que dos destinos pertenecen al mismo mercado hotelero?**

**El problema de la fragmentación**: el dataset contiene ~26.000 nombres de ciudades distintos, con distribución de observaciones muy desigual. El trabajo en turismo hace que se consoliden datos de múltiples proveedores con sus propias estandarizaciones. Esto lleva a tener registros que refieren a la misma ciudad bajo nombres distintos.

**El rol fundamental de `destination_with_nearest.csv`**: este archivo mapea cada ciudad a un `nearest_destination_id` — el ID de un destino canónico cercano que tenga masa suficiente de observaciones. Sin esta consolidación, el análisis carece de la masa estadística necesaria para que cualquier comparación de precios sea confiable.

**Por qué es una hipótesis y no una verdad**: el criterio de agrupación usado es proximidad geográfica — razonable, pero discutible. ¿Miami y Miami Beach comparten realmente la misma dinámica de precios, o tienen perfiles distintos que al mezclarlos se distorsionan mutuamente?

**El mercado va más allá de la geografía**: la temporada del año, el día de la semana, la categoría del hotel o la duración de la estadía pueden ser tan determinantes como la geografía — o más.

### 2. Definición de contexto: ¿qué condiciones hacen que dos precios sean comparables?

El precio de un hotel nunca se evalúa en el vacío. Una habitación a $80/noche en Miami puede ser una ganga en diciembre (temporada alta) y un precio elevado en febrero (temporada baja).

**¿Qué conjunto de variables define que dos búsquedas son suficientemente similares como para comparar sus precios?**

Dimensiones a considerar:
- **Temporal**: mes del año, semana del mes, día de la semana, días especiales o eventos
- **Composición del viaje**: noches de estadía, número de habitaciones, adultos, niños
- **Categoría del alojamiento**: ¿los hoteles más caros de un destino siguen sus propias dinámicas?
- **Condiciones del mercado**: ¿el número de hoteles disponibles (`avg_hotel_count`) afecta los precios observados?

*La definición de CONTEXTO determina directamente la calidad de cualquier algoritmo de detección.*

## Dimensiones candidatas a explorar

No es necesario explorar todas — lo importante es llegar a una **hipótesis razonada** sobre qué combinación produce segmentos internamente coherentes y suficientemente densos como para sostener un análisis estadístico.

| Dimensión | Variable(s) | Pregunta guía |
|-----------|-------------|---------------|
| Localización geográfica | `city`, `state`, `country`, `destination_final` | ¿Los precios son comparables entre destinos cercanos? |
| Día de la semana | derivada de `date_start` | ¿Los precios suben los fines de semana? ¿En todos los destinos por igual? |
| Temporada del año | `month` derivado de `date_start` | ¿La estacionalidad afecta igual a todos los mercados? |
| Categoría del hotel | segmentos de precio (`price_bucket`) | ¿Los hoteles económicos siguen el mismo ciclo que los de lujo? |
| Duración de la estadía | `nights` | ¿Las estadías largas tienen dinámica distinta? |
| Demanda | `count_repeated` | ¿Los destinos con alta demanda tienen precios más estables? |
| Oferta | `avg_hotel_count` | ¿Cuando colapsa la oferta, suben los precios de forma distinta? |

## Preguntas disparadoras

Algunas preguntas para guiar la exploración:

1. ¿Los precios suben los fines de semana?
2. ¿En todos los destinos por igual, o solo en los de ocio?
3. ¿Los hoteles económicos siguen el mismo ciclo estacional que los de lujo?
4. ¿Hay destinos donde la oferta disponible (`avg_hotel_count`) colapsa en ciertas fechas y eso afecta los precios de forma distinta al resto?
5. ¿Qué combinación de variables produce grupos con precios más homogéneos (menor varianza interna)?

## Datos disponibles

| Archivo | Descripción |
|---------|-------------|
| `datos_historicos_2024.csv` | ~2.5M registros de búsquedas hoteleras 2024 (~200 MB) |
| `datos_historicos_2025.csv` | ~2.5M registros de búsquedas hoteleras 2025 (~200 MB) |
| `destination_with_nearest.csv` | Mapping de ~26.000 ciudades a destinos canónicos |

**Importante**: cada fila no representa el precio de un hotel individual, sino el rango estadístico de *todos* los hoteles mostrados en esa búsqueda.

### Columnas principales del dataset

| Columna | Descripción |
|---------|-------------|
| `city`, `state`, `country`, `country_code` | Destino geográfico |
| `date_start`, `date_end`, `nights` | Fechas y duración de la estadía |
| `number_of_rooms`, `number_of_adults`, `number_of_kids` | Composición del viaje |
| `count_repeated` | Cantidad de veces que se realizó esta búsqueda (proxy de demanda) |
| `avg_hotel_count` | Promedio de hoteles disponibles mostrados (proxy de oferta) |
| `avg_price_average` | **Variable central**: precio promedio del conjunto de hoteles en USD |
| `max_price_high`, `min_price_low` | Rango de precios del mercado |

### Datos descargados en: `../data/`

## Entrega esperada

Notebook con:

1. **Visualizaciones** que muestren el comportamiento de los precios a lo largo de las distintas dimensiones
2. **Hipótesis documentada** sobre qué segmentación de mercado propone el grupo — y por qué
3. **Análisis de densidad**: ¿las segmentaciones propuestas generan suficientes observaciones por grupo para ser estadísticamente confiables?

---

## Referencia

El notebook `exploratory_analysis.ipynb` en la raíz del proyecto carga los datos reales, recorre las etapas del pipeline paso a paso y muestra cómo el sistema clasifica un precio. Puede servir como **punto de partida** para la exploración del TP1.

In [ ]:
# Arrancá acá con tu exploración
